# CV Masterclass 01: Convolutional Lineage & Mathematics

Welcome to the first notebook of the Computer Vision Deep Dive Sequence.

We do not jump straight into `torch.nn.Conv2d`. To understand why computer vision models are built the way they are, we must trace the entire lineage of spatial filtering. We will build from Dense Layers, identify their catastrophic failure, invent the 2D Convolution, and evolve it mathematically into Dilated and Depthwise Separable Convolutions.

---

## 🎓 Socratic Deep Check
Ponder this question as we progress:

> *"Standard Convolutions computationally explode when the channel depth gets too large. Why does splitting a single $3\times3$ convolution into a 'Depthwise' spatial filter followed by a 'Pointwise' $1\times1$ filter mathematically reduce the parameters by a factor of 8 or 9, while preserving the exact same accuracy?"*

---

1. **Dense Layers (The Spatial Failure):** parameter explosion and loss of topology.
2. **Standard Convolutions (The Spatial Prior):** Local connectivity, and Weight Sharing.
3. **The Output Shape Formula (Derived):** Mathematical striding footprint.
4. **Padding Modes: Valid, Same, Full:** Manual NumPy implementations.
5. **Grouped Convolutions (The Bridge):** Interpolating across channels.
6. **Dilated (Atrous) Convolutions:** Expanding Receptive Fields freely.
7. **Depthwise Separable Convolutions:** The MobileNet optimization.
8. **Sub-Pixel Convolutions (PixelShuffle):** Trading channel depth for spatial resolution.
9. **Deformable Convolutions:** Kernels that mold to object geometry.
10. **Deep Socratic Synthesis:** Synthesizing striding tradeoffs.
11. **Core Concepts & Pitfalls Summary.**


## 📑 Table of Contents

* [🎓 Socratic Deep Check](#socratic-deep-check)
* [1. Dense Layers (The Spatial Failure)](#1-dense-layers-the-spatial-failure)
  * [⚠️ Common Pitfalls](#common-pitfalls)
* [2. Standard Convolutions (The Spatial Prior)](#2-standard-convolutions-the-spatial-prior)
  * [The Shapes (1D, 2D, 3D)](#the-shapes-1d-2d-3d)
  * [⚠️ Common Pitfalls](#common-pitfalls)
* [3. The Output Shape Formula (Derived)](#3-the-output-shape-formula-derived)
  * [⚠️ Common Pitfalls](#common-pitfalls)
* [4. Padding Modes: Valid, Same, Full](#4-padding-modes-valid-same-full)
  * [⚠️ Common Pitfalls](#common-pitfalls)
* [5. Grouped Convolutions (The Bridge)](#5-grouped-convolutions-the-bridge)
  * [⚠️ Common Pitfalls](#common-pitfalls)
* [6. Dilated (Atrous) Convolutions](#6-dilated-atrous-convolutions)
* [7. Depthwise Separable Convolutions](#7-depthwise-separable-convolutions)
  * [Step A: Depthwise Convolution](#step-a-depthwise-convolution)
  * [Step B: Pointwise Convolution](#step-b-pointwise-convolution)
* [8. Sub-Pixel Convolutions (PixelShuffle)](#8-sub-pixel-convolutions-pixelshuffle)
* [9. Deformable Convolutions (DCNv2)](#9-deformable-convolutions-dcnv2)
* [10. Winograd & CoordConv: The Senior Optimization](#10-winograd-coordconv-the-senior-optimization)
  * [10.1 The Winograd Algorithm](#101-the-winograd-algorithm)
  * [10.2 CoordConv: Solving "Translation Invariance" (The Uber AI Fix)](#102-coordconv-solving-translation-invariance-the-uber-ai-fix)
* [11. 🎓 Deep Socratic Synthesis](#11-deep-socratic-synthesis)
* [12. Core Concepts & Pitfalls Summary](#12-core-concepts-pitfalls-summary)
  * [❌ PITFALL 1: Expecting `Conv2d` to loop over channels.](#pitfall-1-expecting-conv2d-to-loop-over-channels)
  * [❌ PITFALL 2: Designing blindly with Transposed Convolutions.](#pitfall-2-designing-blindly-with-transposed-convolutions)
  * [❌ PITFALL 3: Assuming Image Geometry is Rigid.](#pitfall-3-assuming-image-geometry-is-rigid)


### 🎓 Junior to Senior: Concept Bridge

**Junior:** Memorizes the shape of inputs and outputs through trial and error. Uses default PyTorch strides and padding without knowing why.

**Senior:** Derives receptive fields analytically. Knows that strided convolutions reduce memory constraints and that dilated convolutions exponentially increase receptive field without adding parameters. Treats convolutions as mathematically structured feature extraction.

---


### 🎓 Junior to Senior: Concept Bridge

**Junior:** Memorizes the shape of inputs and outputs through trial and error. Uses default PyTorch strides and padding without knowing why.

**Senior:** Derives receptive fields analytically. Knows that strided convolutions reduce memory constraints and that dilated convolutions exponentially increase receptive field without adding parameters. Treats convolutions as mathematically structured feature extraction.

---


### 🎓 Junior to Senior: Concept Bridge

**Junior:** Memorizes the shape of inputs and outputs through trial and error. Uses default PyTorch strides and padding without knowing why.

**Senior:** Derives receptive fields analytically. Knows that strided convolutions reduce memory constraints and that dilated convolutions exponentially increase receptive field without adding parameters. Treats convolutions as mathematically structured feature extraction.

---


## 1. Dense Layers (The Spatial Failure)

Imagine a tiny $224 \times 224$ RGB image. Flattened, it is a 1D vector of length $150,528$.

If we feed this into a single standard Dense layer with just 1,000 neurons, the Weight Matrix shape is `[150528, 1000]`. That is **150.5 Million parameters** for a single pitiful layer.

**The Loss of Topology:** A dense layer treats the pixel at geometric coordinate `[0, 0]` as completely mathematically unrelated to the pixel right next to it at `[0, 1]`. All concept of 2D space is instantly destroyed when you flatten the image. The model cannot naturally detect corners or edges.

### ⚠️ Common Pitfalls
*   **Assuming flattening is harmless:** Practitioners sometimes flatten spatial data too early in custom architectures, permanently destroying structural relationships that later layers could have exploited.


## 2. Standard Convolutions (The Spatial Prior)

A Convolution solves both flaws of the Dense layer through two rules:
1.  **Local Connectivity:** A neuron is only mathematically allowed to look at a tiny local patch (e.g., $3\times3$) of the previous layer. 
2.  **Weight Sharing:** The exact same $3\times3$ weight kernel is physically dragged across the entire image. If the kernel learns to detect a vertical edge at coordinate `[0,0]`, it automatically perfectly detects a vertical edge at `[200, 200]`. This is called **Translation Invariance**.

### The Shapes (1D, 2D, 3D)
*   **1D Conv:** Used for Audio/Time-Series data. The kernel slides in 1 geometric direction (Time).
*   **2D Conv:** Used for Images. The kernel slides in 2 directions (Width, Height). 
    *   *Note: A 2D Conv physically processes the Color Channels (Depth) simultaneously, but it does NOT "slide" across the channels. It sums them up. Hence, it is 2D, not 3D.*
*   **3D Conv:** Used for Videos or Medical CT Scans. The kernel physically slides across Width, Height, AND Time (or physical Depth in an MRI scan).

### ⚠️ Common Pitfalls
*   **2D vs 3D Convolution confusion:** Believing a 2D Conv on RGB images is a 3D conv because the input has 3 dimensions (W, H, C). A 2D Conv is strictly 2D because it slides in 2 spatial dimensions, computing a dot product covering the full depth `C` at every stop!


## 3. The Output Shape Formula (Derived)

When sliding a kernel of size $K$ across an image of width $W$, we step by a stride $S$, potentially adding $P$ padding pixels to each side.

**The Derivation:**
1. The padded image width is $W + 2P$.
2. The kernel spans $K$ pixels, so there are an initial $(W + 2P - K)$ pixels *remaining* for the kernel's left edge to slide into.
3. If we jump by $S$ pixels at a time, the number of steps we can take is $(W + 2P - K) / S$.
4. We add $+1$ because we have to count the *first* initial placement at index 0.

**The Golden Formula:**  
`output_size = floor( (W - K + 2P) / S ) + 1`

Let's walk through concrete examples:
1. **Classic MNIST Conv:** $W=28$, $K=3$, $P=0$, $S=1$
   - `floor((28 - 3 + 0) / 1) + 1 = 26` -> Output is $26 \times 26$.
2. **Same-Padding (No Shrinkage):** $W=224$, $K=3$, $P=1$, $S=1$
   - `floor((224 - 3 + 2) / 1) + 1 = 224` -> Perfectly preserves $224 \times 224$.
3. **Strided Conv as Pooling Replacement:** $W=224$, $K=3$, $P=0$, $S=2$
   - `floor((224 - 3 + 0) / 2) + 1 = 111` -> Halves the spatial size (approx). Often used with $K=3, P=1, S=2$ to get exactly $112$, mimicking `MaxPool2d`.


In [ ]:
import torch
import torch.nn as nn

# Strided Conv (reduces spatial dims directly directly holding parameters)
strided_conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1, stride=2)

# Standard Conv + MaxPool2d (the old way to reduce spatial dims)
conv_and_pool = nn.Sequential(
    nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1, stride=1),
    nn.MaxPool2d(kernel_size=2, stride=2)
)

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
dummy_image = torch.randn(1, 3, 224, 224) # Batch, Channels, Height, Width

out_strided = strided_conv(dummy_image)
out_pooled = conv_and_pool(dummy_image)

print(f"Original Image: {dummy_image.shape}")
print(f"Strided Conv Output:  {out_strided.shape}")
print(f"Conv+MaxPool Output:  {out_pooled.shape}")

assert out_strided.shape == out_pooled.shape, "Shapes do not match!"


### ⚠️ Common Pitfalls
*   **Off-by-one errors with Even Kernels:** Using an even kernel size (like $2\times2$ or $4\times4$) makes symmetric padding mathematically impossible without shifting the image center. Stick to odd kernels ($3, 5, 7$) with padding $P = (K-1)/2$.


## 4. Padding Modes: Valid, Same, Full

*   **Valid:** No padding at all ($P=0$). The output physically shrinks.
*   **Same:** Padding is added so that if $S=1$, the output width exactly matches the input width.
*   **Full:** Maximum padding ($P=K-1$) so the kernel visits every possible interaction, even just 1 pixel on the edge.

To truly understand how a convolution operates, we will build a vectorized implementation. Professional libraries don't use nested loops; they use matrix tricks like `im2col` or `stride_tricks` to slide the window instantly.


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from numpy.lib.stride_tricks import as_strided

def manual_2d_conv_fast(image, kernel, padding=0, stride=1):
    """
    A vectorized NumPy implementation of a 2D Convolution using sliding window view.
    This is significantly faster than nested loops and represents how matrix 
    libraries (BLAS) approach the problem.
    """
    # 1. Apply padding
    if padding > 0:
        image = np.pad(image, pad_width=padding, mode='constant', constant_values=0)
    
    H, W = image.shape
    Kh, Kw = kernel.shape
    
    # 2. Calculate output shape
    out_H = (H - Kh) // stride + 1
    out_W = (W - Kw) // stride + 1
    
    # 3. Create sliding window view (im2col logic)
    # The 'strides' dictate how many bytes to skip to find the next pixel
    s_h, s_w = image.strides
    windows = as_strided(
        image, 
        shape=(out_H, out_W, Kh, Kw), 
        strides=(s_h * stride, s_w * stride, s_h, s_w)
    )
    
    # 4. Sum of element-wise multiplication across the last two dimensions
    return np.tensordot(windows, kernel, axes=((2, 3), (0, 1)))

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
image_np = np.random.rand(10, 10).astype(np.float32)
kernel_np = np.random.rand(3, 3).astype(np.float32)

# Fast Vectorized Run
out_fast = manual_2d_conv_fast(image_np, kernel_np, padding=1, stride=1)

# PyTorch Comparison
image_th = torch.from_numpy(image_np).unsqueeze(0).unsqueeze(0)
kernel_th = torch.from_numpy(kernel_np).unsqueeze(0).unsqueeze(0)
out_torch = F.conv2d(image_th, kernel_th, padding=1).squeeze().numpy()

print(f"Manual Fast Output Shape: {out_fast.shape}")
print(f"Max Difference vs PyTorch: {np.max(np.abs(out_fast - out_torch)):.6f}")
assert np.allclose(out_fast, out_torch, atol=1e-5)


### ⚠️ Common Pitfalls
*   **"Same" Padding with Strides:** Setting PyTorch padding mode to `'same'` will crash if you use a stride $>1$. "Same" mathematically implies preserving output dimensions, which is impossible if you are skipping pixels.


## 5. Grouped Convolutions (The Bridge)

A standard 2D convolution combines spatial filtering and channel cross-mixing into a single operation. If we have $C_{in}$ channels, the kernel physically evaluates all $C_{in}$ channels simultaneously.

**Grouped Convolutions** act as a mathematical bridge or interpolation between a standard convolution and a depthwise convolution:
- `groups = 1`: Standard convolution. 1 massive group sees all channels.
- `groups = 2`: Input channels are split in half. Two kernels operate independently, one per half.
- `groups = C_in`: **Depthwise Convolution**. Every individual channel gets its own independent spatial kernel. They NEVER mix!

**The Parameter Count Formula:**
- Standard parameters: $C_{in} \times C_{out} \times K \times K$
- Grouped parameters: $\frac{C_{in}}{g} \times C_{out} \times K \times K$


In [ ]:
# Compare Grouped Conv parameter counts
c_in = 512
c_out = 512
kernel_size = 3

conv_standard = nn.Conv2d(c_in, c_out, kernel_size, groups=1, bias=False)
conv_grouped = nn.Conv2d(c_in, c_out, kernel_size, groups=4, bias=False)
conv_depthwise = nn.Conv2d(c_in, c_out, kernel_size, groups=c_in, bias=False)

def get_params(layer):
    return sum(p.numel() for p in layer.parameters())

print(f"Standard (groups=1):    {get_params(conv_standard):,} params")
print(f"Grouped (groups=4):     {get_params(conv_grouped):,} params")
print(f"Depthwise (groups=512): {get_params(conv_depthwise):,} params")

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
dummy_512 = torch.randn(1, 512, 16, 16)
out_dw = conv_depthwise(dummy_512)
print(f"\nDepthwise Output Shape: {out_dw.shape}")

# Verify math exactly matches formula
expected_dw_math = int((c_in / c_in) * c_out * 3 * 3)
assert get_params(conv_depthwise) == expected_dw_math, "Formula mismatch!"


### ⚠️ Common Pitfalls
*   **Confusing `groups` with `out_channels`:** `groups=4` does not output 4 channels. It divides the *input* channels into 4 isolated lanes. The output channels are also divided equally amongst those lanes!


## 6. Dilated (Atrous) Convolutions

If you want to increase an imaging network's "Receptive Field" (how much of the original image a deep neuron can 'see'), you typically use a Max-Pooling layer. This compresses a $224\times224$ map down to $112\times112$. 

**The Problem:** In Semantic Segmentation (where you need a perfectly crisp $224\times224$ mask output), throwing away $75\%$ of your spatial pixels during max pooling is catastrophic.

**The Atrous Fix:** Invented for DeepLab, a Dilated Convolution physically injects empty spaces "holes" into the $3\times3$ kernel. The 9 weights spread out and cover a $5\times5$ spatial area by simply skipping the pixels in between. 
**Result:** The Receptive Field physically triples, but the spatial resolution of the feature map never shrinks. No max-pooling is required!


In [ ]:
def show_dilated_receptive_field(kernel_size, dilation_rate):
    effective_size = kernel_size + (kernel_size - 1) * (dilation_rate - 1)
    trainable_params = kernel_size * kernel_size
    receptive_field = effective_size 
    return trainable_params, effective_size, receptive_field

print(f"{'Rate':<5} | {'Trainable Params':<16} | {'Effective Span':<14} | {'Receptive Field':<15}")
print("-" * 65)
for rate in [1, 2, 3, 4]:
    p, span, rf = show_dilated_receptive_field(kernel_size=3, dilation_rate=rate)
    print(f"{rate:<5} | {p:<16} | {span:<14} | {rf:<15}")


## 7. Depthwise Separable Convolutions

A standard `Conv2d` layer taking 512 channels and outputting 512 channels uses $3 \times 3 \times 512 \times 512 = 2.3 \text{ Million parameters}$.

Google's **MobileNet** architecture revolutionized Edge AI by factoring that monolithic operation into two mutually exclusive steps:

### Step A: Depthwise Convolution
Instead of looking at all 512 input channels at once, we apply one single strictly isolated 2D $3\times3$ filter per channel independently.
**Cost:** $3 \times 3 \times 512 = 4,608 \text{ parameters}$. 

### Step B: Pointwise Convolution
To fix the lack of cross-group intelligence, we apply a massive $1\times1$ Convolution across all 512 channels. 
**Cost:** $1 \times 1 \times 512 \times 512 = 262,144 \text{ parameters}$.

**Total Parameters:** 266,752 (vs 2.3M) → **8.8x efficiency gain**.


In [ ]:
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.depthwise = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1, groups=in_channels, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )
        self.pointwise = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.pointwise(self.depthwise(x))

dummy = torch.randn(2, 512, 14, 14)
ds_module = DepthwiseSeparableConv(512, 512)
print(f"Forward pass Output: {ds_module(dummy).shape}")


## 8. Sub-Pixel Convolutions (PixelShuffle)

In high-resolution vision (Super-Resolution, Generative Models), transposed convolutions cause "checkerboard artifacts." **PixelShuffle** avoids this by producing more channels and rearranging them spatially. 

1. Apply `Conv2d(C_in, C_out * r^2, 3, padding=1)`.
2. Apply `PixelShuffle(r)` to rearrange to `(B, C, r*H, r*W)`.

In [ ]:
class SubPixelConv(nn.Module):
    def __init__(self, in_channels, out_channels, upscale_factor=2):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels * upscale_factor**2,
                                kernel_size=3, padding=1)
        self.pixel_shuffle = nn.PixelShuffle(upscale_factor)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        return self.relu(self.pixel_shuffle(self.conv(x)))

x = torch.randn(2, 64, 14, 14)
upsampler = SubPixelConv(64, 64, upscale_factor=2)
print(f"Upsampled: {upsampler(x).shape}")


## 9. Deformable Convolutions (DCNv2)

Standard kernels are rigid. **Deformable Convolutions** add a learned offset field, allowing the individual sampling points of the kernel to mold to the object's actual contour.


In [ ]:
import torchvision.ops as ops

class DeformableConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, padding=1):
        super().__init__()
        self.conv_offset = nn.Conv2d(in_ch, 2 * kernel_size * kernel_size,
                                     kernel_size, padding=padding)
        self.conv_weight = nn.Parameter(torch.randn(out_ch, in_ch, kernel_size, kernel_size))
        nn.init.kaiming_normal_(self.conv_weight)
        self.padding = padding
        
    def forward(self, x):
        offset = self.conv_offset(x)
        return ops.deform_conv2d(x, offset, self.conv_weight, padding=self.padding)

dcn = DeformableConv2d(64, 128)
dummy = torch.randn(2, 64, 32, 32)
print(f"Deformable Output: {dcn(dummy).shape}")


## 10. Winograd & CoordConv: The Senior Optimization

Before we synthesize, we must look at how convolutions are actually handled in high-performance hardware and how we fix their inherent "Coordinate Blindness."

### 10.1 The Winograd Algorithm
How does a GPU perform millions of $3\times3$ convolutions so fast? It doesn't use the simple sliding math. For small kernels, high-performance libraries use the **Winograd Algorithm**.
- **The Physics**: For a $3\times3$ kernel and $2\times2$ output block, standard convolution requires **36 multiplications**. Winograd transforms the data and kernel into a different mathematical space, reducing the multiplications to just **16**. 
- **The Tradeoff**: It uses more additions and transformations, but since multiplications are the "expensive" operation on silicon, Winograd provides a massive speed boost for mobile and data-center inference.

### 10.2 CoordConv: Solving "Translation Invariance" (The Uber AI Fix)
Convolutions are **Translation Invariant**—they see "a cat" regardless of where it is. But sometimes, we *need* the network to know exactly where a pixel is (e.g., in a coordinate regression task like object detection or navigation).

**The Fix**: Uber AI's **CoordConv** layer simply concatenates two extra channels to the input feature map:
1. An $x$-coordinate channel (0.0 on the left, 1.0 on the right).
2. A $y$-coordinate channel (0.0 on top, 1.0 on bottom).

Now, the $3\times3$ kernel can learn to combine the visual feature with its physical location.


---
## ✅ Knowledge Check

### Q1: Why is a 3x3 convolution preferred over a 7x7 convolution in modern networks?
<details><summary>👉 Answer</summary>

Two stacked 3x3 convolutions have the same receptive field (5x5) as a 5x5 convolution but use fewer parameters and involve more non-linear activations, leading to a deeper, more expressive network.
</details>

### Q2: What is the formula for the output spatial dimension of a Conv2d layer?
<details><summary>👉 Answer</summary>

It is `floor((W_in - Kernel + 2*Padding) / Stride) + 1`.
</details>

---
## 🔨 Practical Practice

### Exercise 1
Calculate Receptive Field: Hand-calculate the receptive field after three sequential 3x3 convolutions with stride 1 and padding 1.

### Exercise 2
Implementation: Write a PyTorch function that manually performs a 2D convolution using purely tensor unrolling (e.g. `im2col` logic) instead of utilizing `F.conv2d`.


## 11. 🎓 Deep Socratic Synthesis

**Q:** *A `Conv2d` with `kernel_size=1, padding=0, stride=2` halves spatial dimensions using only $1 \times C_{in} \times C_{out}$ multiplications per stop. Why do architectures like MobileNetV2 use this rather than a strided `3x3`?*

**Ponder deeply:** delegating spatial understanding to a fast depthwise step followed by a $1\times1$ projection yields massive computational savings while maintaining receptive field capability.


## 12. Core Concepts & Pitfalls Summary

### ❌ PITFALL 1: Expecting `Conv2d` to loop over channels.
A 2D Conv processes all input channels simultaneously. To enforce channel independence, use `groups=in_channels`.

### ❌ PITFALL 2: Designing blindly with Transposed Convolutions.
Always favor **Sub-Pixel Convolutions (PixelShuffle)** or `Upsample + Conv` for premium generative models to avoid checkerboard artifacts.

### ❌ PITFALL 3: Assuming Image Geometry is Rigid.
Objects rotate and deform. Use **Deformable Convolutions** for dynamic receptive fields.